# Strategy Comparison: FedGRA vs. High-Loss

Compare FedGRA and High-Loss test accuracy curves on MNIST and FMNIST.


In [ ]:
import sys
from pathlib import Path
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add repository root for style.py
root_dir = Path("../../..").resolve()
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from style import MatplotlibStyle

MatplotlibStyle().apply()


In [ ]:
# ============================================================
# Load CSV files
# ============================================================

DATASETS = {
    "MNIST": {
        "FedGRA": Path("mnist/fedgra_mnist_min100_-train-20260617_192203-c076d59e.csv"),
        "High-Loss": Path("mnist/high_loss_mnist_-train-20260617_182132-31823659.csv"),
    },
    "FMNIST": {
        "FedGRA": Path("fmnist/fedgra_fmnist_min100_-train-20260617_191039-12dbf5cd.csv"),
        "High-Loss": Path("fmnist/high_loss_fmnist_-train-20260617_181138-d62f9a9e.csv"),
    },
}

def extract_method_name(csv_path):
    with open(csv_path, "r", encoding="utf-8") as f:
        first_line = f.readline().strip()
    if first_line.startswith("Config,"):
        cfg = json.loads(first_line[len("Config,"):])
        return cfg.get("client_selection", {}).get("method", "unknown")
    return "unknown"

def load_training_csv(csv_path):
    with open(csv_path, "r", encoding="utf-8") as f:
        first_line = f.readline()
    header = 1 if first_line.startswith("Config,") else 0
    df = pd.read_csv(csv_path, header=header)
    if "round" in df.columns:
        df = df.set_index("round")
    else:
        df.index = range(len(df))
    return df

learning_data = {}
for dataset_name, files in DATASETS.items():
    learning_data[dataset_name] = {}
    for label, csv_path in files.items():
        if not csv_path.exists():
            raise FileNotFoundError(csv_path)
        method = extract_method_name(csv_path)
        df = load_training_csv(csv_path)
        learning_data[dataset_name][label] = df
        print(
            f"{dataset_name} {label:9s} method={method:12s} "
            f"rounds={len(df)} best_acc={df['accuracy'].max():.4f} "
            f"(r{df['accuracy'].idxmax()}) final_acc={df['accuracy'].iloc[-1]:.4f}"
        )


In [ ]:
# ============================================================
# Plot helpers: Accuracy only
# ============================================================

COLORS = {"FedGRA": "#2a7de1", "High-Loss": "#e53e3e"}

def rolling_mean(series, window):
    return series.rolling(window=window, min_periods=1).mean()

def build_plot_data(data_dict, window=5):
    plot_data = {}
    for name, df in data_dict.items():
        plot_data[name] = {
            "rounds": df.index.to_numpy(),
            "acc_raw": df["accuracy"].to_numpy(),
            "acc_smooth": rolling_mean(df["accuracy"], window).to_numpy(),
        }
    return plot_data

def _draw_strategy_learning_curve_panel(
    ax,
    data_dict,
    title,
    window=5,
    accuracy_ylim=(0, 1.0),
    x_ticks=(0, 25, 50, 75, 100),
    show_ylabel=True,
    show_legend=True,
    legend_loc="lower right",
):
    plot_data = build_plot_data(data_dict, window=window)
    for name, data in plot_data.items():
        color = COLORS[name]
        ax.plot(data["rounds"], data["acc_raw"], color=color, linewidth=0.8, alpha=0.4)
        ax.plot(data["rounds"], data["acc_smooth"], color=color, linewidth=2.2, label=name)
        best_idx = np.argmax(data["acc_raw"])
        ax.scatter(
            data["rounds"][best_idx],
            data["acc_raw"][best_idx],
            color=color,
            marker="*",
            s=200,
            zorder=5,
            edgecolors="black",
            linewidths=0.5,
        )

    ax.set_title(title, fontsize=22)
    ax.set_xlabel("Communication Round", fontsize=20)
    if show_ylabel:
        ax.set_ylabel("Accuracy", fontsize=20)
    ax.set_xlim(0, 100)
    ax.set_xticks(x_ticks)
    ax.set_ylim(accuracy_ylim)
    ax.grid(True, alpha=0.3)
    if show_legend:
        ax.legend(fontsize=15, loc=legend_loc)

def plot_strategy_learning_curves(
    data_dict,
    title,
    save_path=None,
    window=5,
    figsize=(8, 6),
    accuracy_ylim=(0, 1.0),
    x_ticks=(0, 25, 50, 75, 100),
):
    fig, ax = plt.subplots(figsize=figsize)
    _draw_strategy_learning_curve_panel(
        ax,
        data_dict,
        title=title,
        window=window,
        accuracy_ylim=accuracy_ylim,
        x_ticks=x_ticks,
        show_ylabel=True,
        show_legend=True,
    )
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=800, bbox_inches="tight")
    plt.show()

def plot_two_strategy_learning_curves(
    learning_data,
    dataset_names=("MNIST", "FMNIST"),
    save_path=None,
    window=5,
    figsize=(10, 5),
    accuracy_ylim=None,
    x_ticks=(0, 25, 50, 75, 100),
    legend_ncol=2,
    legend_loc="upper center",
    legend_bbox_to_anchor=(0.5, 1.17),
):
    if len(dataset_names) != 2:
        raise ValueError("dataset_names must contain exactly two dataset names.")

    if accuracy_ylim is None:
        accuracy_ylim = [(0, 1.0), (0, 1.0)]
    elif isinstance(accuracy_ylim[0], (int, float)):
        accuracy_ylim = [accuracy_ylim, accuracy_ylim]
    elif len(accuracy_ylim) != 2:
        raise ValueError("accuracy_ylim must be one tuple or two per-panel tuples.")

    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=False)
    handles = labels = None
    for idx, (ax, dataset_name, y_lim) in enumerate(zip(axes, dataset_names, accuracy_ylim)):
        _draw_strategy_learning_curve_panel(
            ax,
            learning_data[dataset_name],
            title=dataset_name,
            window=window,
            accuracy_ylim=y_lim,
            x_ticks=x_ticks,
            show_ylabel=(idx == 0),
            show_legend=False,
        )
        if handles is None:
            handles, labels = ax.get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc=legend_loc,
        bbox_to_anchor=legend_bbox_to_anchor,
        ncol=legend_ncol,
        fontsize=15,
        frameon=False,
    )

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=800, bbox_inches="tight")
    plt.show()


In [ ]:
# ============================================================
# MNIST strategy learning curves
# ============================================================

plot_strategy_learning_curves(
    learning_data["MNIST"],
    title="MNIST Strategy Comparison",
    save_path="mnist_strategy_learning_curves.pdf",
    accuracy_ylim=(0, 0.65),
)


In [ ]:
# ============================================================
# FMNIST strategy learning curves
# ============================================================

plot_strategy_learning_curves(
    learning_data["FMNIST"],
    title="FMNIST Strategy Comparison",
    save_path="fmnist_strategy_learning_curves.pdf",
    accuracy_ylim=(0, 0.5),
)


In [ ]:
# ============================================================
# MNIST and FMNIST strategy learning curves, side by side
# ============================================================

plot_two_strategy_learning_curves(
    learning_data,
    dataset_names=("MNIST", "FMNIST"),
    figsize=(10, 5),
    accuracy_ylim=[(0, 0.65), (0, 0.5)],
    save_path="mnist_fmnist_strategy_learning_curves.pdf",
)
